# US-007：时频分析与 ERP 可视化

**目标：** 绘制高质量 ERP 波形图、脑地形图和时频图，从多维度呈现分析结果。

## 0. 准备数据

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

# 加载 sample 数据并做预处理
sample_dir = mne.datasets.sample.data_path()
raw_fname = sample_dir / "MEG" / "sample" / "sample_audvis_raw.fif"
raw = mne.io.read_raw_fif(raw_fname, preload=True).pick_types(eeg=True, stim=True)
raw.filter(l_freq=1.0, h_freq=40.0)

# 事件和 Epochs
events = mne.find_events(raw, stim_channel='STI 014')
event_id = {'auditory/left': 1, 'auditory/right': 2, 'visual/left': 3, 'visual/right': 4}

epochs = mne.Epochs(
    raw, events, event_id=event_id,
    tmin=-0.5, tmax=1.0,
    baseline=(None, 0),
    preload=True,
    reject=dict(eeg=150e-6),
)
print(f"Epochs: {epochs}")

## 1. Evoked — 蝴蝶图 (Butterfly Plot)

所有通道的 ERP 波形叠加显示，快速了解整体响应。

In [ ]:
evoked_aud = epochs['auditory/left'].average()

# 蝴蝶图
fig = evoked_aud.plot(
    spatial_colors=True,         # 每条线不同颜色
    titles='Auditory Left — Butterfly',
)
# 参数 gfp=True → 叠加绘制 GFP (Global Field Power)

## 2. 脑地形图 (Topomap)

在指定时间点上绘制头皮电位分布。

In [ ]:
# 绘制特定时间点的地形图序列
times = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]  # 秒
fig = evoked_aud.plot_topomap(
    times=times,
    ch_type='eeg',
    time_unit='s',
    colorbar=True,
    ncols=3, nrows='auto',
)

### 2.1 选择特定通道画 ERP

In [ ]:
# 画 Cz 通道（中央顶叶，N100/P300 等经典 ERP 成分常看的位置）
# 用 EEG 030 (sample 数据中近似 Cz)
evoked_aud.pick('EEG 030').plot()

## 3. 多条件 ERP 对比

`plot_compare_evokeds` 是同类型的标准对比方式。

In [ ]:
# 合并左右耳听觉条件
evoked_aud_L = epochs['auditory/left'].average()
evoked_aud_R = epochs['auditory/right'].average()
evoked_vis_L = epochs['visual/left'].average()

# 多条件对比
fig = mne.viz.plot_compare_evokeds(
    dict(
        Auditory_Left=evoked_aud_L,
        Auditory_Right=evoked_aud_R,
        Visual_Left=evoked_vis_L,
    ),
    picks='eeg',           # 只画 EEG
    combine='mean',         # 所有通道平均
    legend='upper right',
    show_sensors='upper left',
    ci=0.95,               # 95% 置信区间（阴影）
    truncate_yaxis='auto',
)

## 4. Joint Plot — 时域+地形图联动

一张图同时显示指定通道的 ERP 波形 + 关键时间点的地形图。

In [ ]:
# Joint plot：显示 Cz 的波形 + 峰值时间点的地形图
evoked_aud_joint = epochs['auditory/left'].average()
fig = evoked_aud_joint.plot_joint(
    times='peaks',         # 自动找峰值
    title='Auditory Left — Joint Plot',
)

## 5. 时频分析

用小波变换或 multitaper 计算时频表示 (TFR)。

In [ ]:
from mne.time_frequency import tfr_morlet

# 选择频率范围和参数
freqs = np.logspace(*np.log10([4, 30]), num=20)  # 对数间隔 4-30 Hz
n_cycles = freqs / 2.    # 频率越低小波周期越少

# 计算时频表示
power = tfr_morlet(
    epochs['auditory/left'],
    freqs=freqs,
    n_cycles=n_cycles,
    use_fft=True,
    return_itc=False,        # 只算功率，不算相位一致性
    decim=3,                 # 降采样加速
    n_jobs=1,
)
print(f"TFR 形状: {power.data.shape}")  # (n_epochs, n_channels, n_freqs, n_times)

### 5.1 时频图：单通道单条件

In [ ]:
# 对 trials 做平均，选一个通道绘制
power_avg = power.average()

# 选 Cz 通道的时频图
fig = power_avg.plot(
    picks=['EEG 030'],
    baseline=(-0.5, 0),         # 基线校正
    mode='logratio',            # 对数变化（dB-like）
    title='Auditory Left — Time-Frequency (Cz)',
    colorbar=True,
)

### 5.2 时频地形图：多通道

In [ ]:
# 绘制多通道时频地形图 (TFR topomap)
# 在指定时间-频率点上显示电位分布
fig = power_avg.plot_topo(
    baseline=(-0.5, 0),
    mode='logratio',
    title='Auditory Left — TFR Topography',
)

### 5.3 条件间时频对比

对比两个条件的时频差异（常用于 ERD/ERS 分析）。

In [ ]:
# 计算听觉右侧的时频
power_R = tfr_morlet(
    epochs['auditory/right'],
    freqs=freqs,
    n_cycles=n_cycles,
    use_fft=True,
    return_itc=False,
    decim=3,
    n_jobs=1,
)

# 差异
power_diff = power.average() - power_R.average()

fig = power_diff.plot(
    picks=['EEG 030'],
    baseline=(-0.5, 0),
    mode='logratio',
    title='Auditory LEFT vs RIGHT — TFR Difference',
)

## 6. ERD/ERS 计算

ERD (Event-Related Desynchronization) / ERS (Synchronization)：
- **ERD**：事件后某频段功率相对于基线的降低（去同步化）
- **ERS**：事件后功率的升高（同步化）

MNE 中 ERD/ERS = TFR 结果的 baseline 归一化（logratio 或百分比变化）。

```python
# baseline=(baseline_tmin, baseline_tmax) 用事件前的基线做归一化
power.plot(baseline=(-0.5, 0), mode='percent')  # 百分比变化
power.plot(baseline=(-0.5, 0), mode='logratio') # dB 变化
```

## 7. 可视化最佳实践

| 图类型 | 适用场景 | 函数 |
|--------|---------|------|
| 蝴蝶图 | 快速浏览所有通道 | `evoked.plot(spatial_colors=True)` |
| 单通道 ERP | 经典波形 | `evoked.pick(ch).plot()` |
| 地形图 | 空间分布 | `evoked.plot_topomap(times=...)` |
| Joint Plot | 论文常用 | `evoked.plot_joint(times='peaks')` |
| 条件对比 | 多条件比较 | `plot_compare_evokeds(...)` |
| 时频图 | 频域动态 | `power.plot(picks=ch, baseline=...)` |

## 8. 练习

1. 尝试不同的频率范围（theta 4-8Hz，alpha 8-13Hz，beta 13-30Hz）
2. 用 `mode='percent'` 重做时频图，和 `'logratio'` 对比
3. 在 `plot_compare_evokeds` 中设置 `ci=False` 去掉置信区间